In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
# -----------------------------
# Decision Tree Function
# -----------------------------
def run_decision_tree_pipeline(file_path, target_column='bug', smote_ratio=0.8):
    df = pd.read_csv(file_path).dropna()
    df[target_column] = df[target_column].apply(lambda x: 0 if x==0 else 1)
    X = df.drop(columns=[target_column])
    y = df[target_column]

    print(f"\n\n=== Decision Tree | Dataset: {file_path} ===")
    print("Original class distribution:")
    print(y.value_counts())

    smote = SMOTE(sampling_strategy=smote_ratio, random_state=42)
    undersample = RandomUnderSampler(sampling_strategy=1.0, random_state=42)
    X_res, y_res = Pipeline([('smote', smote), ('under', undersample)]).fit_resample(X, y)
    print("Resampled class distribution:")
    print(pd.Series(y_res).value_counts())

    # Find best ccp_alpha and max_depth
    clf = DecisionTreeClassifier(random_state=42)
    clf.fit(X_res, y_res)
    path = clf.cost_complexity_pruning_path(X_res, y_res)
    ccp_alphas = path.ccp_alphas

    best_alpha, best_depth, best_score = None, None, 0
    for alpha in ccp_alphas:
        for depth in [4,5,6,7]:
            dt = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha, max_depth=depth, min_samples_leaf=5)
            score = np.mean(cross_val_score(dt, X_res, y_res, cv=5, scoring='f1_macro'))
            if score > best_score:
                best_alpha, best_depth, best_score = alpha, depth, score

    print(f"Best Alpha: {best_alpha:.6f}, Depth: {best_depth}, CV F1: {best_score:.3f}")

    X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, stratify=y_res, random_state=42)
    dt_final = DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha, max_depth=best_depth, min_samples_leaf=5)
    dt_final.fit(X_train, y_train)

    y_pred = dt_final.predict(X_test)
    metrics = {
        'Model': 'Decision Tree',
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'AUC': roc_auc_score(y_test, y_pred),
        'Best_Params': {'ccp_alpha': best_alpha, 'max_depth': best_depth}
    }

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    return dt_final, metrics


# -----------------------------
# Random Forest Function
# -----------------------------
def run_rf_hybrid_pipeline(file_path, target_column='bug', smote_ratio=0.8):
    df = pd.read_csv(file_path).dropna()
    df[target_column] = df[target_column].apply(lambda x: 0 if x==0 else 1)
    X = df.drop(columns=[target_column])
    y = df[target_column]

    print(f"\n\n=== Random Forest | Dataset: {file_path} ===")
    print("Original class distribution:")
    print(y.value_counts())

    smote = SMOTE(sampling_strategy=smote_ratio, random_state=42)
    undersample = RandomUnderSampler(sampling_strategy=1.0, random_state=42)
    rf = RandomForestClassifier(random_state=42, min_samples_leaf=5, n_jobs=-1)

    pipeline = Pipeline([
        ('smote', smote),
        ('under', undersample),
        ('rf', rf)
    ])

    param_grid = {
        'rf__n_estimators': [100, 200],
        'rf__max_depth': [None, 5, 10],
        'rf__criterion': ['gini', 'entropy']
    }

    grid = GridSearchCV(pipeline, param_grid, scoring='f1_macro', cv=5, n_jobs=-1, verbose=1)
    grid.fit(X, y)

    print("\nBest Parameters (GridSearchCV):", grid.best_params_)
    print(f"Best CV F1 (macro): {grid.best_score_:.3f}")

    smote = SMOTE(sampling_strategy=smote_ratio, random_state=42)
    undersample = RandomUnderSampler(sampling_strategy=1.0, random_state=42)
    X_res, y_res = Pipeline([('smote', smote), ('under', undersample)]).fit_resample(X, y)
    print("Resampled class distribution:")
    print(pd.Series(y_res).value_counts())

    best_params = grid.best_params_
    rf_final = RandomForestClassifier(
        n_estimators=best_params['rf__n_estimators'],
        max_depth=best_params['rf__max_depth'],
        criterion=best_params['rf__criterion'],
        random_state=42,
        n_jobs=-1
    )

    X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, stratify=y_res, random_state=42)
    rf_final.fit(X_train, y_train)

    y_pred = rf_final.predict(X_test)
    metrics = {
        'Model': 'Random Forest',
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'AUC': roc_auc_score(y_test, y_pred),
        'Best_Params': best_params
    }

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    return rf_final, metrics


# -----------------------------
# Combined Runner
# -----------------------------
def run_all_models(file_paths):
    results = []
    for fp in file_paths:
        print("\n===============================")
        print(f"Processing file: {fp}")
        print("===============================")
        
        dt_model, dt_metrics = run_decision_tree_pipeline(fp)
        rf_model, rf_metrics = run_rf_hybrid_pipeline(fp)
        
        dt_metrics['Dataset'] = fp
        rf_metrics['Dataset'] = fp
        results.extend([dt_metrics, rf_metrics])

    results_df = pd.DataFrame(results)
    results_df.to_csv("model_results_summary.csv", index=False)
    print("\n✅ All results saved to 'model_results_summary.csv'")
    return results_df


# -----------------------------
# Example usage
# -----------------------------
file_paths = [
    r"C:\Users\sodhi\Desktop\Design Project\combined\df_1.csv",
    r"C:\Users\sodhi\Desktop\Design Project\combined\df_2.csv",
    r"C:\Users\sodhi\Desktop\Design Project\combined\df_3.csv",
    r"C:\Users\sodhi\Desktop\Design Project\combined\df_4.csv",
    r"C:\Users\sodhi\Desktop\Design Project\combined\df_5.csv"
]

results_df = run_all_models(file_paths)
print("\nFinal Summary:")
print(results_df)



Processing file: C:\Users\sodhi\Desktop\Design Project\combined\df_1.csv


=== Decision Tree | Dataset: C:\Users\sodhi\Desktop\Design Project\combined\df_1.csv ===
Original class distribution:
bug
0    1342
1     350
Name: count, dtype: int64
Resampled class distribution:
bug
0    1073
1    1073
Name: count, dtype: int64


C:\Users\sodhi\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\sodhi\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\sodhi\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\sodhi\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\sodhi\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

Best Alpha: 0.000925, Depth: 7, CV F1: 0.769

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.85      0.79       215
           1       0.83      0.71      0.76       215

    accuracy                           0.78       430
   macro avg       0.78      0.78      0.78       430
weighted avg       0.78      0.78      0.78       430



=== Random Forest | Dataset: C:\Users\sodhi\Desktop\Design Project\combined\df_1.csv ===
Original class distribution:
bug
0    1342
1     350
Name: count, dtype: int64
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Best Parameters (GridSearchCV): {'rf__criterion': 'entropy', 'rf__max_depth': 10, 'rf__n_estimators': 200}
Best CV F1 (macro): 0.700
Resampled class distribution:
bug
0    1073
1    1073
Name: count, dtype: int64

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.84      0.85       215
           1       0.84   